In [2]:
!uv pip install sounddevice numpy

Using Python 3.12.13 environment at: C:\Users\eitanturok\good-vibrations\.venv
Resolved 4 packages in 563ms
Installed 1 package in 9ms
 + sounddevice==0.5.5


In [1]:
import sounddevice as sd
import numpy as np

# List devices — find the MOTU ASIO entry
print(sd.query_devices())
print(sd.query_hostapis())  # confirm ASIO is listed

   0 Microsoft Sound Mapper - Input, MME (2 in, 0 out)
>  1 Line In 7-8 (UltraLite-mk5), MME (2 in, 0 out)
   2 Line In 5-6 (UltraLite-mk5), MME (2 in, 0 out)
   3 Mic/Line/Inst In 1-2 (UltraLite, MME (2 in, 0 out)
   4 Line In 3-4 (UltraLite-mk5), MME (2 in, 0 out)
   5 Microsoft Sound Mapper - Output, MME (0 in, 2 out)
<  6 Phones Out 1-2 (UltraLite-mk5), MME (0 in, 2 out)
   7 Realtek Digital Output (Realtek, MME (0 in, 2 out)
   8 Line Out 7-8 (UltraLite-mk5), MME (0 in, 2 out)
   9 PL2792Q (NVIDIA High Definition, MME (0 in, 2 out)
  10 Line Out 5-6 (UltraLite-mk5), MME (0 in, 2 out)
  11 Line Out 9-10 (UltraLite-mk5), MME (0 in, 2 out)
  12 Line Out 3-4 (UltraLite-mk5), MME (0 in, 2 out)
  13 Main Out 1-2 (UltraLite-mk5), MME (0 in, 2 out)
  14 Primary Sound Capture Driver, Windows DirectSound (2 in, 0 out)
  15 Line In 7-8 (UltraLite-mk5), Windows DirectSound (2 in, 0 out)
  16 Line In 5-6 (UltraLite-mk5), Windows DirectSound (2 in, 0 out)
  17 Mic/Line/Inst In 1-2 (UltraLite-mk

In [2]:
def find_device(name_substring, hostapi_name="Windows WASAPI"):
    hostapis = sd.query_hostapis()
    target_hostapi = next(i for i, h in enumerate(hostapis) if h['name'] == hostapi_name)
    for idx, dev in enumerate(sd.query_devices()):
        if (dev['hostapi'] == target_hostapi
            and name_substring in dev['name']
            and dev['max_output_channels'] > 0):
            return idx
    raise ValueError(f"No output device matching {name_substring!r}")

_main   = find_device("Main Out 1-2")
_line34 = find_device("Line Out 3-4")
_line56 = find_device("Line Out 5-6")
_line78 = find_device("Line Out 7-8")
_line910 = find_device("Line Out 9-10")
_phones = find_device("Phones Out 1-2")

SPEAKERS = {
    1:  (_line34, 0, False),
    2:  (_line34, 1, False),
    3:  (_main,   1, False),
    4:  (_main,   0, False),
    5:  (_line56, 1, False),
    6:  (_line56, 0, False),
    7:  (_line910, 0, False),
    8: (_line910, 0, False),
    9:  (_line78, 0, False),
    10:  (_line78, 1, False),
    11: (_phones, 0, True),
}

In [3]:
device_info = sd.query_devices(_main)
SAMPLE_RATE = int(device_info['default_samplerate'])
print(f"Using sample rate: {SAMPLE_RATE}")

Using sample rate: 44100


In [4]:
def make_tone(freq=440, duration=1.0, amplitude=0.2):
    """Generate a sine wave numpy array."""
    t = np.linspace(0, duration, int(SAMPLE_RATE * duration), endpoint=False)
    return (amplitude * np.sin(2 * np.pi * freq * t)).astype(np.float32)


def load_audio(path):
    """Load an audio file as a mono numpy array."""
    audio, _ = sf.read(path, dtype='float32', always_2d=True)
    return audio.mean(axis=1)  # downmix to mono


def play(speaker_key, audio):
    """Play a mono audio array on a single speaker."""
    device_idx, channel_idx, is_mono = SPEAKERS[speaker_key]

    buf = np.zeros((len(audio), 2), dtype=np.float32)
    if is_mono:
        buf[:, 0] = buf[:, 1] = audio
    else:
        buf[:, channel_idx] = audio

    sd.play(buf, samplerate=SAMPLE_RATE, device=device_idx)
    sd.wait()

In [ ]:
tone = make_tone(700, 3)

In [8]:
for speaker in sorted(SPEAKERS):
    print(f'{speaker=}')
    play(speaker, tone)

speaker=1
speaker=2
speaker=3
speaker=4
speaker=5
speaker=6
speaker=7


KeyboardInterrupt: 